In [1]:
# coding: utf-8
import os, gc, glob, json, logging
import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from collections import defaultdict

# ===============================
# ⭐ 模式切换（只改这里）
# ===============================
RUN_MODE = "feature"     # "score" | "feature"

from v1_0_20251201.run_model_feature import (
    generate_score,
    run_model_feature
)

# =====================================================
# 1. 日志控制
# =====================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

# =====================================================
# 🔧 统一任务参数（只改这里）
# =====================================================
TASK_NAME = "old_reloan_intent_1222"
BASE_DIR = "/opt/workspace/aus_group_drive/Eliam/cdaml_bigq"

# =====================================================
# 2. 路径配置
# =====================================================
SAMPLE_PATH = f"{BASE_DIR}/data/{TASK_NAME}/samples.csv"
TXN_DIR = f"{BASE_DIR}/modeling/tmp_eliam_{TASK_NAME}_variable_illion_transaction_middle"
BAL_DIR = f"{BASE_DIR}/modeling/tmp_eliam_{TASK_NAME}_balance_time_series_middle"
OUT_PATH = f"{BASE_DIR}/data/{TASK_NAME}/sample_with_feature.csv"
ERROR_PATH = f"{BASE_DIR}/data/{TASK_NAME}/sample_with_feature_error_detail.csv"

CHUNK_SIZE = 500_000

# =====================================================
# 3. 工具函数
# =====================================================
def nan_to_empty(x):
    return "" if pd.isna(x) else x


def is_valid_float(x):
    try:
        float(x)
        return True
    except Exception:
        return False


def is_valid_balance_record(rec: dict) -> bool:
    return is_valid_float(rec.get("balance"))

# =====================================================
# 4. 索引构建
# =====================================================
def build_csv_index(csv_dir):
    logger.info(f"📚 构建索引：{csv_dir}")
    index = defaultdict(set)

    for fp in glob.glob(os.path.join(csv_dir, "*.csv")):
        try:
            for chunk in pd.read_csv(
                fp,
                usecols=["user_id", "application_id"],
                chunksize=CHUNK_SIZE
            ):
                for k in zip(chunk.user_id, chunk.application_id):
                    index[k].add(fp)
        except Exception as e:
            logger.warning(f"⚠️ 索引失败跳过：{fp} | {e}")

    logger.info(f"✅ 索引完成：{len(index)} keys")
    return index

# =====================================================
# 5. 读取匹配行
# =====================================================
def load_match_rows_from_index(csv_index, csv_dir, user_id, application_id):
    fps = csv_index.get((user_id, application_id))
    if not fps:
        return pd.DataFrame()

    dfs = []
    for fp in fps:
        try:
            df = pd.read_csv(fp)
            m = df[
                (df["user_id"] == user_id) &
                (df["application_id"] == application_id)
            ]
            if not m.empty:
                dfs.append(m)
        except Exception:
            continue

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# =====================================================
# 6. 构造模型输入
# =====================================================
def build_input_data(r, txn_index, bal_index):
    uid, aid = r["user_id"], r["application_id"]
    ft = str(r["sample_datetime"])

    txn_df = load_match_rows_from_index(txn_index, TXN_DIR, uid, aid)
    bal_df = load_match_rows_from_index(bal_index, BAL_DIR, uid, aid)

    txn_records = (
        txn_df[
            [
                "amount", "balance", "bank_account_id", "category",
                "dr_cr", "illion_trx_uuid", "text", "third_party",
                "transaction_date", "transaction_id", "trx_type"
            ]
        ].applymap(nan_to_empty).to_dict("records")
        if not txn_df.empty else []
    )

    bal_records = []
    if not bal_df.empty:
        raw_bal = (
            bal_df[
                ["balance", "balance_date", "balance_id", "bank_account_id"]
            ].applymap(nan_to_empty).to_dict("records")
        )
        for rec in raw_bal:
            if is_valid_balance_record(rec):
                bal_records.append(rec)

    return {
        "userId": uid,
        "applicationId": aid,
        "flowTime": ft,
        "illion_raw_transactions": txn_records,
        "illion_day_end_balances": bal_records
    }

# =====================================================
# 7. 单样本执行
# =====================================================
def process_one_sample(args):
    r, txn_index, bal_index = args
    try:
        input_data = build_input_data(r, txn_index, bal_index)

        # ===============================
        # ⭐ 根据模式调用
        # ===============================
        if RUN_MODE == "feature":
            res = run_model_feature(input_vars=input_data)
        else:
            res = generate_score(input_vars=input_data)

        out = dict(r)

        # ===============================
        # ⭐ 统一展开返回结果
        # ===============================
        if isinstance(res, dict):
            # feature 模式：{"features": {...}}
            if RUN_MODE == "feature" and "features" in res:
                if isinstance(res["features"], dict):
                    out.update(res["features"])
            else:
                # score 模式
                for k, v in res.items():
                    if isinstance(v, dict):
                        out.update(v)
                    else:
                        out[k] = v

        return out

    except Exception as e:
        out = dict(r)
        out["feature_error"] = str(e)
        out["_error_detail"] = {
            "user_id": r["user_id"],
            "application_id": r["application_id"],
            "error_message": str(e),
            "input_data_json": json.dumps(
                build_input_data(r, txn_index, bal_index),
                ensure_ascii=False
            )
        }
        return out

    finally:
        gc.collect()

# =====================================================
# 8. 主流程
# =====================================================
def main():
    logger.info("🚀 加载 samples")
    df = pd.read_csv(SAMPLE_PATH)
    df["sample_datetime"] = df["sample_datetime"].astype(str)

    done_keys = set()
    if os.path.exists(OUT_PATH):
        logger.info("🔁 启用断点续跑")
        done_df = pd.read_csv(OUT_PATH, usecols=["user_id", "application_id"])
        done_keys = set(zip(done_df.user_id, done_df.application_id))

    records = [
        r for r in df.to_dict("records")
        if (r["user_id"], r["application_id"]) not in done_keys
    ]

    if not records:
        logger.info("✅ 无待跑样本")
        return

    txn_index = build_csv_index(TXN_DIR)
    bal_index = build_csv_index(BAL_DIR)

    n_workers = max(cpu_count() - 4, 1)
    write_header = not os.path.exists(OUT_PATH)
    write_err_header = not os.path.exists(ERROR_PATH)

    with open(OUT_PATH, "a", encoding="utf-8", newline="") as fout, \
         open(ERROR_PATH, "a", encoding="utf-8", newline="") as ferr:

        with Pool(n_workers) as pool:
            for r in tqdm(
                pool.imap_unordered(
                    process_one_sample,
                    [(r, txn_index, bal_index) for r in records]
                ),
                total=len(records),
                desc="Processing samples"
            ):
                error_detail = r.pop("_error_detail", None)

                pd.DataFrame([r]).to_csv(
                    fout,
                    header=write_header,
                    index=False
                )
                write_header = False
                fout.flush()

                if error_detail:
                    pd.DataFrame([error_detail]).to_csv(
                        ferr,
                        header=write_err_header,
                        index=False
                    )
                    write_err_header = False
                    ferr.flush()

    logger.info(f"✅ 完成：{OUT_PATH}")
    logger.info(f"⚠️ 错误明细：{ERROR_PATH}")

if __name__ == "__main__":
    main()

2025-12-28 06:39:00,326 | INFO | 🚀 加载 samples
2025-12-28 06:39:48,392 | INFO | 📚 构建索引：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/tmp_eliam_old_reloan_intent_1222_variable_illion_transaction_middle
2025-12-28 06:42:33,929 | INFO | ✅ 索引完成：112123 keys
2025-12-28 06:42:33,930 | INFO | 📚 构建索引：/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/tmp_eliam_old_reloan_intent_1222_balance_time_series_middle
2025-12-28 06:43:25,106 | INFO | ✅ 索引完成：112638 keys
Processing samples:   0%|          | 49/112716 [03:38<147:42:39,  4.72s/it]Process ForkPoolWorker-28:
Process ForkPoolWorker-21:
Processing samples:   0%|          | 49/112716 [03:39<140:16:19,  4.48s/it]Process ForkPoolWorker-20:
Process ForkPoolWorker-26:
Process ForkPoolWorker-23:
Process ForkPoolWorker-27:
Process ForkPoolWorker-25:
Process ForkPoolWorker-24:
Process ForkPoolWorker-18:
Process ForkPoolWorker-17:

Process ForkPoolWorker-22:
Process ForkPoolWorker-19:
Process ForkPoolWorker-14:
Process ForkPoolWorker-13:

KeyboardInterrupt: 

In [ ]:
# 版本二